In [ ]:
%pip install -U langgraph langchain langchain-core langchain-openai langchain-community tavily-python

In [3]:
import os
import operator

from dotenv import load_dotenv
from typing import TypedDict, Annotated

from azure.identity import DefaultAzureCredential, get_bearer_token_provider

from langgraph.graph import StateGraph, END

from langchain_core.messages import (
    AnyMessage,
    HumanMessage,
    ToolMessage,
    SystemMessage,
)

from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults

# =========================================================
# Setup Azure OpenAI
# =========================================================

load_dotenv()

openai_endpoint = os.getenv("ENDPOINT")
model_deployment = os.getenv("MODEL_DEPLOYMENT")

token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default"
)

model = ChatOpenAI(
    model=model_deployment,
    base_url=openai_endpoint,
    api_key=token_provider(),
    temperature=0,
)

# =========================================================
# Search Tool
# =========================================================

tool = TavilySearchResults(max_results=4, tavily_api_key="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")

# =========================================================
# State Definition
# =========================================================

class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

# =========================================================
# Agent
# =========================================================

class Agent:

    def __init__(self, model, tools, system=""):

        self.system = system

        self.tools = {t.name: t for t in tools}

        self.model = model.bind_tools(tools)

        graph = StateGraph(AgentState)

        graph.add_node("llm", self.call_model)
        graph.add_node("action", self.take_action)

        graph.add_conditional_edges(
            "llm",
            self.exists_action,
            {
                True: "action",
                False: END
            }
        )

        graph.add_edge("action", "llm")

        graph.set_entry_point("llm")

        self.graph = graph.compile()

    def exists_action(self, state):

        result = state["messages"][-1]

        return len(result.tool_calls) > 0

    def call_model(self, state):

        messages = state["messages"]

        if self.system:
            messages = [
                SystemMessage(content=self.system)
            ] + messages

        response = self.model.invoke(messages)

        return {
            "messages": [response]
        }

    def take_action(self, state):

        tool_calls = state["messages"][-1].tool_calls

        results = []

        for t in tool_calls:

            print("\nCalling Tool:")
            print(t)

            if t["name"] not in self.tools:

                result = "Bad tool name. Please retry."

            else:

                result = self.tools[t["name"]].invoke(
                    t["args"]
                )

            results.append(
                ToolMessage(
                    tool_call_id=t["id"],
                    name=t["name"],
                    content=str(result)
                )
            )

        print("\nBack to LLM...\n")

        return {
            "messages": results
        }

# =========================================================
# System Prompt
# =========================================================

system_prompt = """
You are a smart research assistant.

Use Tavily search whenever information must be verified.

You may perform multiple searches before answering.

Always provide a final answer.
"""

# =========================================================
# Build Agent
# =========================================================

abot = Agent(
    model,
    [tool],
    system=system_prompt
)

# =========================================================
# Run Query
# =========================================================

query = """
Who won the Super Bowl in 2024?

In which state is the winning team's headquarters located?

What is the GDP of that state?

Answer each question.
"""

result = abot.graph.invoke(
    {
        "messages": [
            HumanMessage(content=query)
        ]
    }
)

print("\n==============================")
print("FINAL ANSWER")
print("==============================\n")

print(result["messages"][-1].content)


Calling Tool:
{'name': 'tavily_search_results_json', 'args': {'query': 'Super Bowl 2024 winner'}, 'id': 'call_GtxQDsrC2WJ0MuIxPkaava7f', 'type': 'tool_call'}

Calling Tool:
{'name': 'tavily_search_results_json', 'args': {'query': 'Kansas City Chiefs headquarters location state'}, 'id': 'call_fQLARGAmEjux465yOrYkNkDx', 'type': 'tool_call'}

Calling Tool:
{'name': 'tavily_search_results_json', 'args': {'query': 'Missouri GDP 2023 nominal BEA state GDP 2023 Missouri GDP'}, 'id': 'call_4Ioad2jh8JYF5Lxa9Ezdhybc', 'type': 'tool_call'}

Back to LLM...


Calling Tool:
{'name': 'tavily_search_results_json', 'args': {'query': 'Missouri GDP 2023 nominal GDP BEA Missouri GDP 2023 current-dollar'}, 'id': 'call_ekH0lWhg9W1i487tlRczMrWi', 'type': 'tool_call'}

Back to LLM...


Calling Tool:
{'name': 'tavily_search_results_json', 'args': {'query': "Missouri GDP 2023 BEA current-dollar 'Missouri GDP 2023' 'current-dollar' 'BEA' 'state GDP 2023 Missouri'"}, 'id': 'call_eGyasEDbO1sEHgdrb1vxPCxK', 'type'